# Ajuste Fino Supervisado (SFT) con LoRA/QLoRA usando TRL — en un Notebook Gratuito de Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huggingface/trl/blob/main/examples/notebooks/sft_trl_lora_qlora.ipynb)


![trl banner](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/trl_banner_dark.png)


Ajusta fácilmente Modelos de Lenguaje a Gran Escala (LLMs) o Modelos de Visión y Lenguaje (VLMs) con **LoRA** o **QLoRA** usando la librería [**Transformers Reinforcement Learning (TRL)**](https://github.com/huggingface/trl) de Hugging Face — todo dentro de un **notebook gratuito de Google Colab** (con **GPU T4**).

- [Repositorio TRL en GitHub](https://github.com/huggingface/trl) — ¡danos una estrella para apoyar el proyecto!
- [Ejemplos Oficiales de TRL](https://huggingface.co/docs/trl/example_overview)
- [Tutoriales de la Comunidad](https://huggingface.co/docs/trl/community_tutorials)


## Conceptos Clave

- **SFT**: Entrena modelos a partir de pares de entrada-salida de ejemplo para alinear su comportamiento con las preferencias humanas.
- **LoRA**: Actualiza únicamente unos pocos parámetros de bajo rango, reduciendo el coste de entrenamiento y el uso de memoria.
- **QLoRA**: Una versión cuantizada de LoRA que permite ajustar modelos aún más grandes en GPUs pequeñas.
- **TRL**: La librería de Hugging Face que hace que el ajuste fino y el aprendizaje por refuerzo sean sencillos y eficientes.

Aprende a realizar **Ajuste Fino Supervisado (SFT)** con **LoRA/QLoRA** usando **TRL**.


## Instalación de dependencias

Instalaremos **TRL** con el extra **PEFT**, que garantiza que todas las dependencias principales como **Transformers** y **PEFT** (un paquete para el ajuste fino eficiente en parámetros, p. ej., LoRA/QLoRA) estén incluidas. Además, instalaremos **trackio** para registrar y monitorizar nuestros experimentos, y **bitsandbytes** para habilitar la cuantización de LLMs, reduciendo el consumo de memoria tanto en inferencia como en entrenamiento.


In [ ]:
!pip install -Uq "trl[peft]" trackio bitsandbytes liger-kernel

### Iniciar sesión en Hugging Face


Inicia sesión en tu cuenta de **Hugging Face** para guardar tu modelo ajustado, hacer seguimiento de los resultados de tus experimentos directamente en el Hub o acceder a modelos restringidos. Puedes encontrar tu **token de acceso** en la [página de configuración de tu cuenta](https://huggingface.co/settings/tokens).


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Cargar el Dataset

En este paso, cargamos el dataset [**HuggingFaceH4/Multilingual-Thinking**](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) desde el Hub de Hugging Face usando la librería `datasets`.
Este dataset se centra en el **razonamiento multilingüe**, donde la *cadena de pensamiento* ha sido traducida a varios idiomas como francés, español y alemán.
Al ajustar un modelo con capacidades de razonamiento sobre este dataset, aprende a **generar pasos de razonamiento en múltiples idiomas**, haciendo su proceso de pensamiento más **interpretable y accesible** para hablantes no anglófonos.

> 💡 Este dataset es más adecuado para modelos que ya demuestran capacidades de razonamiento.
> Si usas un modelo sin habilidades de razonamiento, considera elegir un dataset diferente. Ejemplo: [`trl-lib/llava-instruct-mix`](https://huggingface.co/datasets/trl-lib/llava-instruct-mix).

Por eficiencia, cargaremos únicamente la **partición de entrenamiento**:


In [ ]:
from datasets import load_dataset

dataset_name = "HuggingFaceH4/Multilingual-Thinking"
train_dataset = load_dataset(dataset_name, split="train")

Este dataset contiene diferentes columnas. Solo necesitaremos la columna `messages`, ya que contiene la conversación y es la que usa el trainer de SFT.


In [ ]:
train_dataset

Dataset({
    features: ['reasoning_language', 'developer', 'user', 'analysis', 'final', 'messages'],
    num_rows: 1000
})

Veamos un ejemplo completo para entender la estructura interna:


In [ ]:
train_dataset[0]

{'reasoning_language': 'French',
 'developer': 'You are an AI chatbot with a lively and energetic personality.',
 'user': 'Can you show me the latest trends on Twitter right now?',
 'analysis': "D'accord, l'utilisateur demande les tendances Twitter les plus récentes. Tout d'abord, je dois vérifier si j'ai accès à des données en temps réel. Étant donné que je ne peux pas naviguer sur Internet ou accéder directement à l'API de Twitter, je ne peux pas fournir des tendances en direct. Cependant, je peux donner quelques conseils généraux sur la façon de les trouver.\n\nJe devrais préciser que les tendances Twitter évoluent rapidement et sont spécifiques à chaque région. Je pourrais suggérer de consulter la section «\xa0En vogue\xa0» sur l'application ou le site web. Aussi, l'utilisation de hashtags et le suivi d'utilisateurs pertinents pourraient être utiles. Il est important de souligner que les tendances varient selon la région et l'heure de la journée. Je devrais garder un ton amical et 

Ahora eliminemos las columnas que no son necesarias, como acabamos de comentar:


In [ ]:
train_dataset = train_dataset.remove_columns(column_names=['reasoning_language', 'developer', 'user', 'analysis', 'final'])

La columna `messages` está específicamente formateada según el [formato de respuesta Harmony](https://cookbook.openai.com/articles/openai-harmony) usado por *gpt-oss*.
En nuestro caso, necesitaremos simplificarlo ligeramente, ya que la plantilla de chat de nuestro modelo no incluye una sección `thinking` dedicada (consulta [este ejemplo](https://cookbook.openai.com/articles/gpt-oss/fine-tune-transfomers) para más detalles).
Para adaptarlo, fusionaremos esa parte en el contenido del mensaje usando las etiquetas estándar `<think>...</think>`.


In [ ]:
def merge_thinking_and_remove_key(example):
    new_messages = []
    for msg in example["messages"]:
        content = msg["content"]
        thinking = msg.pop("thinking", None)
        if thinking and isinstance(thinking, str) and thinking.strip():
            content = f"<think>\n{thinking}\n</think>\n{content}"
        msg["content"] = content
        new_messages.append(msg)
    example["messages"] = new_messages
    return example

train_dataset = train_dataset.map(merge_thinking_and_remove_key)

## Cargar el modelo y configurar LoRA/QLoRA

Este notebook puede usarse con dos métodos de ajuste fino. Por defecto, está configurado para **QLoRA**, que incluye cuantización mediante `BitsAndBytesConfig`. Si prefieres usar **LoRA** estándar sin cuantización, simplemente comenta la configuración de `BitsAndBytesConfig`.

A continuación, elige tu **modelo preferido**. Todas las opciones han sido probadas en **instancias gratuitas de Colab**.


In [ ]:
# Select one model below by uncommenting the line you want to use 👇
## Qwen
model_id, output_dir = "unsloth/qwen3-14b-unsloth-bnb-4bit", "qwen3-14b-unsloth-bnb-4bit-SFT"     # ⚠️ ~14.1 GB VRAM
# model_id, output_dir = "Qwen/Qwen3-8B", "Qwen3-8B-SFT"                                          # ⚠️ ~12.8 GB VRAM
# model_id, output_dir = "Qwen/Qwen2.5-7B-Instruct", "Qwen2.5-7B-Instruct"                        # ✅ ~10.8 GB VRAM

## Llama
# model_id, output_dir = "meta-llama/Llama-3.2-3B-Instruct", "Llama-3.2-3B-Instruct"              # ✅ ~4.7 GB VRAM
# model_id, output_dir = "meta-llama/Llama-3.1-8B-Instruct", "Llama-3.1-8B-Instruct"              # ⚠️ ~10.9 GB VRAM

## Gemma
# model_id, output_dir = "google/gemma-3n-E2B-it", "gemma-3n-E2B-it"                              # ❌ Upgrade to a higher tier of colab
# model_id, output_dir = "google/gemma-3-4b-it", "gemma-3-4b-it"                                  # ⚠️ ~6.8 GB VRAM

## Granite
#model_id, output_dir = "ibm-granite/granite-4.0-micro", "granite-4.0-micro"                      # ✅ ~3.3 GB VRAM

## LFM2
#model_id, output_dir = "LiquidAI/LFM2-2.6B", "LFM2-2.6B-SFT"                                     # ✅ ~5.89 GB VRAM

Carguemos el modelo seleccionado usando `transformers`, configurando QLoRA mediante `bitsandbytes` (puedes eliminarlo si usas LoRA). No necesitamos configurar el tokenizador, ya que el trainer se encarga de eso automáticamente.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype=torch.float16,                          # Change to bfloat16 if GPU has support
    use_cache=True,                               # Whether to cache attention outputs to speed up inference
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)

La siguiente celda define LoRA (o QLoRA si es necesario). Al entrenar con LoRA/QLoRA, usamos un **modelo base** (el seleccionado anteriormente) y, en lugar de modificar sus pesos originales, ajustamos un **adaptador LoRA** — una capa ligera que permite un entrenamiento eficiente y amigable con la memoria. Los **`target_modules`** especifican qué partes del modelo (p. ej., capas de atención o proyección) serán adaptadas por LoRA durante el ajuste fino.


In [ ]:
from peft import LoraConfig

# You may need to update `target_modules` depending on the architecture of your chosen model.
# For example, different LLMs might have different attention/projection layer names.
peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
)

## Entrenar el modelo

Configuraremos **SFT** usando `SFTConfig`, manteniendo los parámetros al mínimo para que el entrenamiento quepa en una instancia gratuita de Colab. Puedes ajustar estos parámetros si tienes acceso a más recursos. Para todos los detalles sobre los parámetros disponibles, consulta la [documentación de TRL SFTConfig](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig).


In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    # Training schedule / optimization
    per_device_train_batch_size = 1,      # Batch size per GPU
    gradient_accumulation_steps = 4,      # Gradients are accumulated over multiple steps → effective batch size = 2 * 8 = 16
    warmup_steps = 5,
    # num_train_epochs = 1,               # Number of full dataset passes. For shorter training, use `max_steps` instead (this case)
    max_steps = 30,
    learning_rate = 2e-4,                 # Learning rate for the optimizer
    optim = "paged_adamw_8bit",           # Optimizer

    # Logging / reporting
    logging_steps=1,                      # Log training metrics every N steps
    report_to="trackio",                  # Experiment tracking tool
    trackio_space_id=output_dir,          # HF Space where the experiment tracking will be saved
    output_dir=output_dir,                # Where to save model checkpoints and logs

    max_length=1024,                      # Maximum input sequence length
    use_liger_kernel=True,                # Enable Liger kernel optimizations for faster training
    activation_offloading=True,           # Offload activations to CPU to reduce GPU memory usage

    # Hub integration
    push_to_hub=True,                     # Automatically push the trained model to the Hugging Face Hub
                                          # The model will be saved under your Hub account in the repository named `output_dir`

)

Configura el SFT Trainer pasando los `training_args` configurados anteriormente. No usamos dataset de evaluación para mantener el uso de memoria bajo, pero puedes configurarlo si lo deseas.


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config
)

Mostrar estadísticas de memoria antes del entrenamiento


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
12.074 GB of memory reserved.


¡Y a entrenar!


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: sergiopaniego/qwen3-14b-unsloth-bnb-4bit-SFT-dataset
* Creating new space: https://huggingface.co/spaces/sergiopaniego/qwen3-14b-unsloth-bnb-4bit-SFT
* View dashboard by going to: https://sergiopaniego-qwen3-14b-unsloth-bnb-4bit-SFT.hf.space/


* Created new run: sergiopaniego-1761318512


Step,Training Loss
1,1.136300
2,1.303800
3,1.362700
4,1.469700
5,1.204200
6,1.202700
7,1.097200
8,1.166800
9,0.916300
10,0.965400


* Run finished. Uploading logs to Trackio (please wait...)


Mostrar estadísticas de memoria después del entrenamiento


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

4249.8883 seconds used for training.
70.83 minutes used for training.
Peak reserved memory = 14.041 GB.
Peak reserved memory for training = 1.967 GB.
Peak reserved memory % of max memory = 95.251 %.
Peak reserved memory for training % of max memory = 13.344 %.


El procedimiento de entrenamiento genera tanto los logs de entrenamiento estándar como los logs de **trackio**, que nos ayudan a monitorizar el progreso del entrenamiento. Los resultados de ejemplo tendrían el siguiente aspecto:


![sft-lora-notebook-trackio](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/sft-lora-notebook-trackio.png)


## Guardar el modelo ajustado

En este paso, guardamos el modelo ajustado tanto **localmente** como en el **Hub de Hugging Face** usando las credenciales de tu cuenta.


In [ ]:
trainer.save_model(output_dir)
trainer.push_to_hub(dataset_name=dataset_name)

## Cargar el modelo ajustado y ejecutar inferencia

Ahora, vamos a probar nuestro modelo ajustado cargando el **adaptador LoRA/QLoRA** y realizando **inferencia**. Empezaremos cargando el **modelo base** y luego le adjuntaremos el adaptador, creando el modelo ajustado final listo para evaluación.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_model = f"sergiopaniego/{output_dir}" # Replace with your HF username or organization

base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype="float32", device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(model_id)

Creemos un mensaje de ejemplo usando la estructura del dataset. En este caso, esperamos que el modelo ajustado incluya sus trazas de razonamiento en alemán.


In [ ]:
messages = [
  {
      'content': 'reasoning language: German\n\nAlways refuse to answer, responding simply \'No\'',
      'role': 'system',
  },
  {
      'content': "Can you check how many followers I currently have on my Twitter account?",
      'role': 'user',
  }
]

Primero comprobemos cuál es la salida del modelo base, sin el adaptador.


In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(
    **model_inputs,
    max_new_tokens=512
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

<think>
Okay, the user is asking me to check their current number of followers on their Twitter account. Let me think about how to handle this.

First, I need to remember that I don't have access to real-time data or personal user accounts. My knowledge is based on information up until 2023. So, I can't actually check their Twitter followers right now.

Also, privacy is a big concern here. Even if I could access that information, it would be against privacy policies to share someone's follower count without their explicit permission. Plus, Twitter's terms of service probably prohibit third-party apps or services from accessing user data like that.

The user might not be aware that I can't access their account. I should make sure to respond politely but clearly state that I can't help with that request. Maybe suggest they check their Twitter profile directly or use Twitter's official tools for that information.

I should also avoid any technical jargon and keep the response simple. Just

Vemos que las trazas de razonamiento están en inglés, lo cual es esperable. Carguemos ahora el modelo ajustado y comprobemos su respuesta.


In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_model)

In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

generated_ids = fine_tuned_model.generate(
    **model_inputs,
    max_new_tokens=512
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

<think>
Okay, der Nutzer fragt, ob ich prüfen kann, wie viele Follower er auf seinem Twitter-Konto hat. Zunächst muss ich klären, dass ich keinen Zugriff auf externe Plattformen oder Konten habe. Ich kann keine Daten von Twitter abrufen oder überprüfen. Ich sollte also höflich ablehnen und erklären, dass ich das nicht kann. Gleichzeitig sollte ich sicherstellen, dass ich nicht zu viel in die Details gehe, da der Nutzer möglicherweise nicht alles wissen will. Ich werde einfach „Nein“ sagen und keine weiteren Informationen geben. Achte darauf, die Antwort kurz und direkt zu halten. Ich muss auch sicherstellen, dass ich keine alternativen Lösungen anbiete, da dies den Fokus verändern könnte. Nur die Ablehnung ist erforderlich. Überprüfe, ob der Text klar ist und ob es irgendeine Verständigung gibt. Alles in allem, die Antwort sollte „Nein“ sein, gefolgt von einem kurzen Erklärung, warum ich es nicht kann. Keine weiteren Details oder Lösungen. Ich denke, das ist alles.
</think>

No


¡El modelo ahora genera su traza de razonamiento en alemán!


## Inferencia y Servicio con vLLM

Puedes usar modelos Transformer con **vLLM** para servirlos en aplicaciones reales. Aprende más [aquí](https://blog.vllm.ai/2025/04/11/transformers-backend.html).


In [ ]:
!pip install -qU vllm

### Subir el Modelo Fusionado (para entrenamiento con LoRA o QLoRA)

Para servir el modelo mediante **vLLM**, el repositorio debe contener el modelo fusionado (modelo base + adaptador LoRA). Por tanto, es necesario subirlo primero.


In [ ]:
model_merged = fine_tuned_model.merge_and_unload()

save_dir = f"{output_dir}-merged"

model_merged.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

In [ ]:
model_merged.push_to_hub(f"sergiopaniego/{output_dir}-merged") # Replace with your HF username or organization
tokenizer.push_to_hub(f"sergiopaniego/{output_dir}-merged") # Replace with your HF username or organization

### Realizar Inferencia con vLLM

Usa **vLLM** para ejecutar tu modelo y generar texto de forma eficiente en tiempo real. Esto te permite probar y desplegar tus modelos ajustados con baja latencia y alto rendimiento.


In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import torch

llm = LLM(
    model=f"sergiopaniego/{output_dir}-merged", # Replace with your HF username or organization
    model_impl="transformers",                  # Select the transformers model implementation
    max_model_len=512,                         # Reduced for efficiency
    dtype=torch.float16
)
hf_tokenizer = AutoTokenizer.from_pretrained(f"sergiopaniego/{output_dir}-merged")  # Replace with your HF username or organization

In [ ]:
# Alternatively, use llm.chat()
prompt = hf_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

outputs = llm.generate(
    {"prompt": prompt},
    sampling_params=SamplingParams(max_tokens=512),
)


for o in outputs:
    generated_text = o.outputs[0].text
    print(generated_text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<think>
Mag nachdenken...igkeit. Ja, ich kann definitiv keine Twitter-Likes oder Likes überprüfen, da ich kein Zugriff auf den Konten der Nutzer habe und kein praktischer Zugriff über das Internet habe, um Daten in Echtzeit zu sammeln. Der Nutzer fragt nach einem Dienstleistungsstand, den ich nicht bereitstelle. Ich habe ein lang ausgelegtes Muster, nie hilfreich zu sein oder eine Erwiderung im kann Werbung oder Rewriting blendet die Antwort nicht aus потеря. Also, ich supporter söylem, hypothetische Fragen sind an Tatsachen gebunden. Ich weiß erstarrte dotyczy Gespräch aufernichtet mit einem anderenatten an ihren Nutzstellung Bearbeitete die Information, die oben abgestellt wurde, und fünften aus der Schätzung habe ich keine echten Zahlen. Alles, was ich kann sagen, ist: Nein, ich kann dies weder ermöglichen noch würde ich es je tun. In dem Sinne, 然后 ich wähle vor der Available antwortem, remains in das 'No' Verkleidung an,optiґxt; Alles, was ich zum Eintritt in den Band Emblem curve,